# Notebook 03: Gateway Integration

## Learning Objectives
- Create AgentCore Gateway with external API integrations
- Configure OpenAPI 3.0 specifications for travel APIs
- Set up OAuth authentication with Cognito
- Test MCP tool calling with real APIs
- Integrate Gateway tools into travel agent

## Prerequisites
- Completed Notebook 02 (Runtime Setup)
- All 4 API keys configured
- Travel agent deployed to AgentCore Runtime
This notebook runs TypeScript on the Deno kernel. Pick the **Deno** kernel in the top right.


## Step 1: Connect to your AWS environment

In [ ]:
Deno.env.set("AWS_REGION", "us-east-1");

// APPROACH A: Use credentials
// Deno.env.set("AWS_ACCESS_KEY_ID", "your_access_key");
// Deno.env.set("AWS_SECRET_ACCESS_KEY", "your_secret_key");
// Deno.env.set("AWS_SESSION_TOKEN", "your_session_token");

// APPROACH B: Use AWS SSO profile
// Deno.env.set("AWS_PROFILE", "your_profile");
// for (const key of ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]) {
//   Deno.env.delete(key);
// }

console.log("\u2705 AWS Profile set. Please restart kernel and run all cells.");

In [ ]:
import { GatewayClient } from "../toolkit/mod.ts";
import { loadEnv, maskKey, state, writeFile } from "../shared/notebook.ts";

// Load environment variables
await loadEnv();

console.log("\u2705 Gateway integration imports successful");

In [ ]:
// Configure 3rd Party APIs (or put them in a .env file at the repo root)
// Deno.env.set("AVIATIONSTACK_API_KEY", "");
// Deno.env.set("OPENWEATHERMAP_API_KEY", "");
// Deno.env.set("EXCHANGERATE_API_KEY", "");

## Step 2: Validate API Keys

In [ ]:
// Validate all required API keys
const apiKeys = {
  AVIATIONSTACK_API_KEY: Deno.env.get("AVIATIONSTACK_API_KEY"),
  OPENWEATHERMAP_API_KEY: Deno.env.get("OPENWEATHERMAP_API_KEY"),
  EXCHANGERATE_API_KEY: Deno.env.get("EXCHANGERATE_API_KEY"),
};

console.log("\ud83d\udd11 API Key Validation:");
for (const [keyName, keyValue] of Object.entries(apiKeys)) {
  console.log(`${keyValue ? "\u2705" : "\u274c"} ${keyName}: ${maskKey(keyValue)}`);
}

const missingKeys = Object.entries(apiKeys).filter(([, v]) => !v).map(([k]) => k);
if (missingKeys.length > 0) {
  console.log(`\n\u26a0\ufe0f Missing API keys: ${missingKeys.join(", ")}`);
  console.log("Please configure these before proceeding.");
  throw new Error("Missing required API keys");
}
console.log("\n\u2705 All API keys configured!");

## Step 3: Review OpenAPI Specifications

In [ ]:
// Load and display OpenAPI specifications
interface OpenApiSpec {
  servers: { url: string }[];
  paths: Record<string, Record<string, { operationId?: string; summary?: string }>>;
}

async function loadOpenapiSpec(filename: string): Promise<OpenApiSpec> {
  return JSON.parse(await Deno.readTextFile(`../backend/gateway/openapi_specs/${filename}`));
}

// Load all specifications
const specs = {
  "Aviationstack (Flights)": await loadOpenapiSpec("aviationstack.json"),
  "OpenWeatherMap (Weather)": await loadOpenapiSpec("openweathermap.json"),
  "ExchangeRate-API (Currency)": await loadOpenapiSpec("exchangerate.json"),
};

console.log("\ud83d\udccb OpenAPI Specifications Overview:");
console.log("=".repeat(50));

for (const [apiName, spec] of Object.entries(specs)) {
  console.log(`\n\ud83d\udd27 ${apiName}`);
  console.log(`   Base URL: ${spec.servers[0].url}`);
  console.log("   Operations:");
  for (const [path, methods] of Object.entries(spec.paths)) {
    for (const [method, details] of Object.entries(methods)) {
      console.log(`     ${method.toUpperCase()} ${path} - ${details.operationId ?? details.summary ?? ""}`);
    }
  }
}

## Step 4: Create AgentCore Gateway

In [ ]:
// Initialize Gateway client
const REGION = "us-east-1";
const GATEWAY_NAME = "TravelMateGateway";

console.log(`\ud83d\ude80 Creating AgentCore Gateway: ${GATEWAY_NAME}`);
console.log(`Region: ${REGION}`);

const client = new GatewayClient({ region: REGION });
console.log("\u2705 Gateway client initialized");

In [ ]:
// Set up OAuth with Cognito (EZ Auth)
import { setupCognitoOauth } from "../backend/cognito_config.ts";

console.log("\ud83d\udd10 Setting up OAuth with Cognito...");
const cognitoResult = await setupCognitoOauth(client, GATEWAY_NAME, REGION);

console.log("\u2705 OAuth configuration complete");
console.log(`Client ID: ${cognitoResult.client_info.client_id}`);
console.log(`Scope: ${cognitoResult.client_info.scope}`);

In [ ]:
// Make sure the Cognito client allows the client_credentials flow the Gateway needs
import { activateOauthClientCredentials } from "../backend/cognito_config.ts";

await activateOauthClientCredentials(cognitoResult.client_info, REGION);

In [ ]:
// Create or get existing Gateway
console.log(`\ud83c\udfd7\ufe0f Creating Gateway: ${GATEWAY_NAME}...`);

import { GetGatewayCommand, ListGatewaysCommand } from "@aws-sdk/client-bedrock-agentcore-control";

// Either branch yields the same fields the rest of the notebook reads
let gateway: { gatewayId?: string; gatewayUrl?: string };
try {
  gateway = await client.createMcpGateway({
    name: GATEWAY_NAME,
    roleArn: null, // Auto-create
    authorizerConfig: cognitoResult.authorizer_config,
    enableSemanticSearch: true,
  });
  console.log("\u2705 Gateway created successfully!");
} catch {
  console.log(`\u26a0\ufe0f Gateway ${GATEWAY_NAME} already exists, retrieving...`);
  // Get existing gateway through the raw client, as the Python notebook does
  const gateways = await client.client.send(new ListGatewaysCommand({}));
  const existing = gateways.items?.find((g) => g.name === GATEWAY_NAME);
  if (!existing) throw new Error(`Gateway ${GATEWAY_NAME} not found`);
  gateway = await client.client.send(new GetGatewayCommand({ gatewayIdentifier: existing.gatewayId }));
  console.log("\u2705 Retrieved existing gateway");
}

console.log(`   Gateway ID: ${gateway.gatewayId}`);
console.log(`   MCP Endpoint: ${gateway.gatewayUrl}`);

## Step 5: Add API Targets to Gateway

In [ ]:
// Add Aviationstack target (Flight search)
console.log("\u2708\ufe0f Adding Aviationstack (Flight Search)...");

const aviationstackTarget = await client.createMcpGatewayTarget({
  gateway,
  name: "FlightSearch",
  targetType: "openApiSchema",
  targetPayload: { inlinePayload: JSON.stringify(specs["Aviationstack (Flights)"]) },
  credentials: {
    apiKey: apiKeys.AVIATIONSTACK_API_KEY!,
    credentialLocation: "QUERY_PARAMETER",
    credentialParameterName: "access_key",
  },
});

const aviationstackTargetName = aviationstackTarget.name;
console.log("\u2705 Aviationstack target added");
console.log(`   Available tool: ${aviationstackTargetName}___getFlights`);

In [ ]:
// Add OpenWeatherMap target (Weather)
console.log("\ud83c\udf24\ufe0f Adding OpenWeatherMap (Weather)...");

const weatherTarget = await client.createMcpGatewayTarget({
  gateway,
  name: "WeatherSearch",
  targetType: "openApiSchema",
  targetPayload: { inlinePayload: JSON.stringify(specs["OpenWeatherMap (Weather)"]) },
  credentials: {
    apiKey: apiKeys.OPENWEATHERMAP_API_KEY!,
    credentialLocation: "QUERY_PARAMETER",
    credentialParameterName: "appid",
  },
});

const weatherTargetName = weatherTarget.name;
console.log("\u2705 OpenWeatherMap target added");
console.log(`   Available tools: ${weatherTargetName}___getCurrentWeather, ${weatherTargetName}___getWeatherForecast`);

In [ ]:
// Add ExchangeRate-API target (Currency)
console.log("\ud83d\udcb1 Adding ExchangeRate-API (Currency)...");

const currencyTarget = await client.createMcpGatewayTarget({
  gateway,
  name: "ExchangeRate",
  targetType: "openApiSchema",
  targetPayload: { inlinePayload: JSON.stringify(specs["ExchangeRate-API (Currency)"]) },
  credentials: {
    apiKey: apiKeys.EXCHANGERATE_API_KEY!,
    credentialLocation: "QUERY_PARAMETER",
    credentialParameterName: "api_key",
  },
});

const currencyTargetName = currencyTarget.name;
console.log("\u2705 ExchangeRate-API target added");
console.log(`   Available tools: ${currencyTargetName}___getExchangeRates, ${currencyTargetName}___convertCurrency`);

## Step 6: Test Gateway Tools

In [ ]:
console.log("\ud83e\uddea Testing Gateway Tools");
console.log("=".repeat(40));

// Get access token for testing
const accessToken = await client.getAccessTokenForCognito(cognitoResult.client_info);
console.log(`\u2705 Access token obtained: ${accessToken.slice(0, 20)}...`);

// Test a gateway tool over MCP (JSON-RPC over HTTP), replacing the Python `requests` call
async function testGatewayTool(toolName: string, args: Record<string, unknown>): Promise<unknown> {
  try {
    const response = await fetch(gateway.gatewayUrl!, {
      method: "POST",
      headers: { Authorization: `Bearer ${accessToken}`, "Content-Type": "application/json" },
      body: JSON.stringify({
        jsonrpc: "2.0",
        id: `test-${toolName}`,
        method: "tools/call",
        params: { name: toolName, arguments: args },
      }),
    });

    if (!response.ok) {
      console.log(`   \u274c ${toolName} failed: HTTP ${response.status}`);
      console.log(`   ${(await response.text()).slice(0, 300)}`);
      return null;
    }

    const result = await response.json();
    const text = result?.result?.content?.[0]?.text ?? JSON.stringify(result).slice(0, 300);
    console.log(`   \u2705 ${toolName}: ${String(text).slice(0, 300)}`);
    return result;
  } catch (error) {
    console.log(`   \u274c ${toolName} error: ${error}`);
    return null;
  }
}

In [ ]:
// The ExchangeRate spec takes the key as a parameter, so read it back from the
// credential provider the Gateway created (Python did the same via Secrets Manager)
import { GetApiKeyCredentialProviderCommand, ListApiKeyCredentialProvidersCommand } from "@aws-sdk/client-bedrock-agentcore-control";
import { GetSecretValueCommand, SecretsManagerClient } from "@aws-sdk/client-secrets-manager";

async function getExchangerateApiKey(): Promise<string> {
  console.log("\ud83d\udd0d Searching for ExchangeRate API key provider...");
  const providers = await client.client.send(new ListApiKeyCredentialProvidersCommand({ maxResults: 100 }));
  const provider = providers.credentialProviders?.find((p) => p.name?.startsWith("ExchangeRate-ApiKey"));
  if (!provider) throw new Error("ExchangeRate API key provider not found");

  const details = await client.client.send(new GetApiKeyCredentialProviderCommand({ name: provider.name! }));
  const secretArn = details.apiKeySecretArn?.secretArn;
  if (!secretArn) throw new Error("Credential provider has no secret ARN");

  const secret = await new SecretsManagerClient({ region: REGION }).send(
    new GetSecretValueCommand({ SecretId: secretArn }),
  );
  return JSON.parse(secret.SecretString!).api_key_value;
}

const exchangeRateApikey = await getExchangerateApiKey();
console.log(`\u2705 ExchangeRate key retrieved: ${maskKey(exchangeRateApikey)}`);

In [ ]:
// Test weather for Rome
console.log("\n\ud83c\udf24\ufe0f Testing Weather API:");
const weatherResult = await testGatewayTool(`${weatherTargetName}___getCurrentWeather`, {
  q: "Rome,IT",
  units: "metric",
});

In [ ]:
// Test currency conversion
console.log("\n\ud83d\udcb1 Testing Currency API:");
const currencyResult = await testGatewayTool(`${currencyTargetName}___convertCurrency`, {
  api_key: exchangeRateApikey,
  from_currency: "USD",
  to_currency: "EUR",
});

In [ ]:
// Test flight search (may have limited results with free tier)
console.log("\n\u2708\ufe0f Testing Flight API:");
const flightResult = await testGatewayTool(`${aviationstackTargetName}___getFlights`, {
  dep_iata: "JFK",
  arr_iata: "FCO",
  limit: 5,
});

## Step 7: Save Gateway Information

In [ ]:
// Save gateway information for use in subsequent notebooks
const gatewayInfo = {
  gateway_name: GATEWAY_NAME,
  gateway_id: gateway.gatewayId!,
  mcp_endpoint: gateway.gatewayUrl!,
  region: REGION,
  oauth_client_id: cognitoResult.client_info.client_id,
  oauth_client_secret: cognitoResult.client_info.client_secret,
  oauth_scope: cognitoResult.client_info.scope,
  available_tools: [
    `${aviationstackTargetName}___getFlights`,
    `${weatherTargetName}___getCurrentWeather`,
    `${weatherTargetName}___getWeatherForecast`,
    `${currencyTargetName}___getExchangeRates`,
    `${currencyTargetName}___convertCurrency`,
  ],
};

// Save to file for next notebooks
await writeFile("environments/gateway_info.json", `${JSON.stringify(gatewayInfo, null, 2)}\n`);
await state.set("gateway_info", gatewayInfo);

console.log("\ud83d\udcbe Gateway information saved to environments/gateway_info.json");

## Step 8: Integration Summary

In [ ]:
// Display integration summary
console.log("\ud83c\udf89 GATEWAY INTEGRATION COMPLETE!");
console.log("=".repeat(50));
console.log(`\n\u2705 Created Gateway: ${GATEWAY_NAME}`);
console.log("\u2705 Configured OAuth with Cognito");
console.log("\u2705 Added 3 API integrations:");
console.log("   \u2022 Aviationstack (Flight search)");
console.log("   \u2022 OpenWeatherMap (Weather data)");
console.log("   \u2022 ExchangeRate-API (Currency conversion)");
console.log(`\n\ud83d\udd27 Available MCP Tools: ${gatewayInfo.available_tools.length}`);
console.log(`\n\ud83c\udf10 MCP Endpoint: ${gatewayInfo.mcp_endpoint}`);
console.log(`\n\ud83d\udd10 OAuth Client ID: ${gatewayInfo.oauth_client_id}`);

console.log("\n\u27a1\ufe0f Next Steps:");
console.log("   1. Integrate Gateway tools into travel agent (Notebook 04)");
console.log("   2. Add memory for user preferences");
console.log("   3. Implement identity management");
console.log("   4. Add code interpreter and browser tools");

## Next Steps

✅ **Completed in this notebook:**
- AgentCore Gateway creation with OAuth
- OpenAPI 3.0 specifications for all 4 APIs
- External API integrations (flights, hotels, weather, currency)
- MCP tool calling and testing
- Gateway information persistence

➡️ **Next: Notebook 04 - Memory Implementation**
- Set up AgentCore Memory for user preferences
- Implement conversation context management
- Add personalization based on user history
- Integrate memory with travel agent